In [ ]:
using Plots
using SparseArrays
using BenchmarkTools

In [ ]:

k(f) = (2 * π * f) / (3 * 10^8)

"""
function next_u(u_index, h, f_grid, u_grid; k = 50)
    i,j = u_index
    north = (i + 1) > 80 ? u_grid[1,j] : u_grid[i+1,j]
    south = (i - 1) < 1 ? u_grid[80,j] : u_grid[i-1,j]
    east = (j + 1) > 100 ? u_grid[i,1] : u_grid[i,j+1]
    west = (j - 1) < 1 ? u_grid[i,100] : u_grid[i,j-1]
    neighbors = (north + south + east + west)
    return (neighbors - f_grid[i,j] * h^2) / (4 - k^2 * h^2)
end
"""

function f_x_y(x_r, y_r, stepsize; A= 10^4, sigma = 0.2)
    foo(x,y) = A * exp(-(((x-x_r)^2 + (y-y_r)^2)/(2 * sigma^2)));
    return [foo(x,y) for x in 0:stepsize:8-stepsize, y in 0:stepsize:10-stepsize]
end

function inside_walls_mask()
    discretisations = Int(1/0.005)
    mask = zeros(8*discretisations,10*discretisations)

    #inner walls 
    mask[586:615, 1:600] .= 2 
    mask[586:615, 800:1199] .= 2
    mask[586:615, 1400:end-1] .= 2

    mask[1:400, 486:515] .= 2
    mask[1:300, 1386:1415] .= 2
    mask[500:599, 1386:1415] .= 2
    mask[601:end, 1186:1215] .= 2

    #edges
    mask[1:30, :] .= 1
    mask[end-29:end, :] .= 1
    mask[:, 1:30] .= 1
    mask[:, end-29:end] .= 1

    return mask
end

function find_mask_val(index, h_grid, mask; h_mask = 0.005)
    i, j = index
    scalar = h_grid / h_mask           # e.g. 0.1 / 0.005 = 20.0

    # map coarse cell center to fine index range [1 .. rows_mask]
    # using (i-0.5)*h_grid / h_mask + 0.5 would be more physically accurate,
    # but a simple block mapping is OK:
    i_scaled = clamp(Int(round((i-0.5) * scalar)), 1, size(mask,1))
    j_scaled = clamp(Int(round((j-0.5) * scalar)), 1, size(mask,2))

    return mask[i_scaled, j_scaled]
end


function construct_matrix(u_grid, mask; k0 = 50.0, h)
    I = Int[]
    J = Int[]
    V = ComplexF64[]
    rows, cols = size(u_grid)
    N = rows * cols

    n_air  = 1.0 + 0.0im
    n_wall = 2.5 + 0.5im    # wall material (inner or thick part of outer wall)

    for j in 1:cols
        for i in 1:rows
            p = i + (j-1)*rows
                        
            cell_type = find_mask_val((i,j), h, mask)  # 0: air, 1: outer wall region, 2: inner wall

            is_outer_wall = (cell_type == 1)

            # Check if this wall cell actually touches interior (air or inner wall)
            # in a cardinal direction.
            has_interior_neighbour = false
            ii = 0; jj = 0

            if is_outer_wall
                for (di, dj) in ((-1,0), (1,0), (0,-1), (0,1))
                    i2 = i + di
                    j2 = j + dj
                    if 1 <= i2 <= rows && 1 <= j2 <= cols && find_mask_val((i2,j2), h, mask) != 1
                        has_interior_neighbour = true
                        ii, jj = i2, j2
                        break
                    end
                end
            end

            if is_outer_wall && has_interior_neighbour
                # -------- Impedance BC on the wall–interior interface --------
                # ∂u/∂n ≈ (u_wall - u_int)/h,  ∂u/∂n - i*k0*u_wall = 0
                # => (1/h - i*k0)*u_wall - (1/h)*u_int = 0

                q = ii + (jj-1)*rows

                push!(I, p); push!(J, p); push!(V, (1/h - im*k0))
                push!(I, p); push!(J, q); push!(V, -1/h)

            else
                # -------- Interior node (air, inner wall, or thick outer wall) --------
                # Use standard Helmholtz stencil with appropriate k_local.

                # Treat any non-air as wall material for k_local:
                n_local = (cell_type == 0) ? n_air : n_wall
                k_local = n_local * k0

                # Center
                push!(I, p)
                push!(J, p)
                push!(V, (k_local^2 * h^2 - 4))

                # Non-periodic neighbours (if inside domain)
                if i > 1
                    north_idx = p - 1
                    push!(I, p); push!(J, north_idx); push!(V, 1.0 + 0.0im)
                end
                if i < rows
                    south_idx = p + 1
                    push!(I, p); push!(J, south_idx); push!(V, 1.0 + 0.0im)
                end
                if j > 1
                    west_idx = p - rows
                    push!(I, p); push!(J, west_idx); push!(V, 1.0 + 0.0im)
                end
                if j < cols
                    east_idx = p + rows
                    push!(I, p); push!(J, east_idx); push!(V, 1.0 + 0.0im)
                end
            end
        end
    end

    return sparse(I, J, V, N, N)
end

function build_rhs(f_grid, h)
    println(size(f_grid))
    rows, cols = size(f_grid)
    b = zeros(ComplexF64, rows*cols)
    for j in 1:cols, i in 1:rows
        p = i + (j-1)*rows
        b[p] = f_grid[i,j] * h^2
    end
    return b
end




m = inside_walls_mask()
heatmap(m)
display(current())




In [ ]:
h= 0.01
u_grid = zeros(Float64, Int(8/h), Int(10/h))
f_field = f_x_y(5.5,2.5, h)
A = construct_matrix(u_grid, m; k0=10, h=h)
b = build_rhs(f_field, h)
u_vec = A \ b

u = reshape(u_vec, size(u_grid))

U = abs.(u)
U ./= maximum(U)              # normalize 0..1
U_dB = 20 .* log10.(U .+ 1e-12)  # avoid log10(0)

p = heatmap(U_dB; clims=(-50, 0))   # dB scale like the assignment

ymin, ymax = ylims(p)
positions = range(ymin, ymax, length=9)
labels = round.(Int, positions ./ ymax .* 8)
yticks!(p, positions, string.(labels))
xmin, xmax = xlims(p)
positions = range(xmin, xmax, length=11)
labels = round.(Int, positions ./ xmax .* 10)
xticks!(p, positions, string.(labels))
display(p)

In [ ]:
@benchmark (construct_matrix(u_grid; k=19, h=h))